💡 **Environment:** `clamp-analyses`

# Description

Builds the paired, identically-scored prediction frame for the cross-method AUROC/AUPRC significance test.

This notebook reads the raw drug-disease prediction HDF5 files from all four methods — **NB06** (single-gene baseline, `gene_based`), **NB07** (ARCHS4 module model, `module_based_archs4`), **NB08** (GTEx, `module_based_gtex`), and **NB09** (recount2, `module_based_recount2`) — and reproduces NB10's exact aggregation so every method is scored on an identical set of (drug, disease) pairs:

1. Per file: rank `score` over the full DOID distribution, then inner-merge with the gold standard.
2. Average ranks across the 5 `n_top_genes` thresholds (per trait, drug, method, tissue).
3. Take the max across the 49 tissues (per trait, drug, method).

Output: a paired frame `predictions_paired.pkl` (4 methods × 685 pairs) consumed by `01_paired_bootstrap_test.ipynb`.

> This notebook does **not** modify NB 06–13 or `libs/`. It **fails loud** if any of NB06–NB09 is missing or incomplete (NB08/NB09 must be run separately before the full grid can be aggregated).

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm

from pyprojroot import here

# Settings

In [3]:
N_TISSUES = 49

# Method -> canonical name (mirrors NB10). NB06-NB09 already write the canonical
# names ('gene_based', 'module_based_archs4', 'module_based_gtex',
# 'module_based_recount2'), so these display-name aliases are only a defensive
# fallback (the .get() default passes canonical names through).
# The bare 'Module-based' alias is intentionally omitted: it is ambiguous across
# datasets (archs4/gtex/recount2) and would misclassify GTEx/recount2 files as
# ARCHS4 in the full 4-method grid.
METHOD_RENAME = {
    'Gene-based':              'gene_based',
    'Module-based (ARCHS4)':   'module_based_archs4',
    'Module-based (GTEx)':     'module_based_gtex',
    'Module-based (recount2)': 'module_based_recount2',
}

# All four methods in the full grid (gene baseline + three module models).
METHOD_THRESHOLDS = {
    'gene_based':            [-1.0, 50.0, 100.0, 250.0, 500.0],
    'module_based_archs4':   [-1.0, 5.0, 10.0, 25.0, 50.0],
    'module_based_gtex':     [-1.0, 5.0, 10.0, 25.0, 50.0],
    'module_based_recount2': [-1.0, 5.0, 10.0, 25.0, 50.0],
}
EXPECTED_METHODS = tuple(METHOD_THRESHOLDS)

In [4]:
DATA_DIR = here('data/drug_disease_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

# Raw prediction HDF5 dirs for the four methods (NB06, NB07, NB08, NB09).
_PRED_BASE = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations')
PREDICTIONS_DIRS = {
    'gene_based':
        _PRED_BASE / '06_prediction_single_gene_based' / 'lincs' / 'predictions' / 'dotprod_neg',
    'module_based_archs4':
        _PRED_BASE / '07_prediction_module_based_archs4' / 'lincs' / 'predictions' / 'dotprod_neg',
    'module_based_gtex':
        _PRED_BASE / '08_prediction_module_based_gtex' / 'lincs' / 'predictions' / 'dotprod_neg',
    'module_based_recount2':
        _PRED_BASE / '09_prediction_module_based_recount2' / 'lincs' / 'predictions' / 'dotprod_neg',
}
# Map each method to the prediction notebook that produces it (for fail-loud msgs).
_METHOD_SOURCE_NB = {
    'gene_based':            'NB06 (06_prediction_single_gene_based)',
    'module_based_archs4':   'NB07 (07_prediction_module_based_archs4)',
    'module_based_gtex':     'NB08 (08_prediction_module_based_gtex)',
    'module_based_recount2': 'NB09 (09_prediction_module_based_recount2)',
}
for name, d in PREDICTIONS_DIRS.items():
    display((name, d))
    # Fail loud (do not silently score on partial data): the full 4-method grid
    # needs all of NB06-NB09 on disk. NB08/NB09 must be run separately first.
    assert d.exists(), (
        f'{name} predictions missing -- run {_METHOD_SOURCE_NB[name]} first: {d}')

OUTPUT_DIR = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/14_signif_test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations')

('gene_based',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based/lincs/predictions/dotprod_neg'))

('module_based_archs4',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg'))

('module_based_gtex',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg'))

('module_based_recount2',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg'))

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/14_signif_test')

# Load PharmacotherapyDB gold standard

In [5]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())
display(gold_standard['true_class'].value_counts())

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


true_class
1    755
0    243
Name: count, dtype: int64

# Helpers (copied verbatim from NB10)

In [6]:
def _get_tissue(data_value):
    """Extract tissue name from the metadata 'data' field."""
    prefix = 'spredixcan-mashr-zscores-'
    assert data_value.startswith(prefix), data_value
    tissue = data_value[len(prefix):]
    for suffix in (
        '-projection-archs4',
        '-projection-gtex',
        '-projection-recount2',
        '-projection',
        '-data',
    ):
        if tissue.endswith(suffix):
            return tissue[:-len(suffix)]
    raise ValueError(f'Cannot extract tissue from metadata data value: {data_value}')

# Load drug-disease predictions

In [7]:
current_prediction_files = []
for d in PREDICTIONS_DIRS.values():
    current_prediction_files.extend(sorted(d.glob('*.h5')))
current_prediction_files.sort()
display(len(current_prediction_files))

980

In [8]:
# Load all prediction files, rank scores, merge with gold standard (NB10 logic).
predictions = []
skipped_files = []

for f in tqdm(current_prediction_files, ncols=100):
    metadata = pd.read_hdf(f, key='/metadata')
    method_name = METHOD_RENAME.get(
        metadata['method'].values[0], metadata['method'].values[0])
    if method_name not in METHOD_THRESHOLDS:
        skipped_files.append((f.name, method_name))
        continue

    # Rank within the full DOID distribution, then keep gold-standard pairs.
    prediction_data = pd.read_hdf(f, key='/prediction')
    prediction_data['score'] = prediction_data['score'].rank()
    prediction_data = pd.merge(
        prediction_data, gold_standard, on=['trait', 'drug'], how='inner')
    prediction_data['trait'] = prediction_data['trait'].astype('category')
    prediction_data['drug'] = prediction_data['drug'].astype('category')

    prediction_data = prediction_data.assign(method=method_name)
    prediction_data['method'] = pd.Categorical(
        prediction_data['method'], categories=EXPECTED_METHODS, ordered=True)
    prediction_data = prediction_data.assign(
        n_top_genes=metadata['n_top_genes'].values[0])

    data_value = metadata['data'].values[0]
    prediction_data = prediction_data.assign(data=data_value)
    prediction_data['data'] = prediction_data['data'].astype('category')
    prediction_data = prediction_data.assign(tissue=_get_tissue(data_value))

    predictions.append(prediction_data)

display(f'Skipped files: {len(skipped_files)}')
if skipped_files:
    display(skipped_files[:10])


  0%|                                                                       | 0/980 [00:00<?, ?it/s]

  0%|                                                               | 1/980 [00:00<03:30,  4.65it/s]

  0%|▏                                                              | 2/980 [00:00<03:13,  5.05it/s]

  0%|▏                                                              | 3/980 [00:00<03:09,  5.15it/s]

  0%|▎                                                              | 4/980 [00:00<03:02,  5.36it/s]

...

100%|████████████████████████████████████████████████████████████▉| 978/980 [02:52<00:00,  5.77it/s]

100%|████████████████████████████████████████████████████████████▉| 979/980 [02:52<00:00,  5.78it/s]

100%|█████████████████████████████████████████████████████████████| 980/980 [02:52<00:00,  5.83it/s]

100%|█████████████████████████████████████████████████████████████| 980/980 [02:52<00:00,  5.68it/s]

'Skipped files: 0'

In [9]:
predictions = pd.concat(predictions, ignore_index=True)
display(predictions.shape)
display(predictions.head())

(671300, 8)

,trait,drug,score,true_class,method,n_top_genes,data,tissue
0,DOID:0050741,DB00215,103099.5,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
1,DOID:0050741,DB00704,355210.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
2,DOID:0050741,DB00822,388169.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
3,DOID:10283,DB00014,80190.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
4,DOID:10283,DB00175,232448.0,0,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous


## Validation checks (enforces NB06–NB09 completeness)

In [10]:
assert not predictions.isna().any().any()

_method_counts = predictions['method'].value_counts().reindex(EXPECTED_METHODS)
display(_method_counts)

N_PREDICTIONS = predictions[['drug', 'trait']].drop_duplicates().shape[0]
display(f'Unique drug-disease pairs: {N_PREDICTIONS}')

for method_name, thresholds in METHOD_THRESHOLDS.items():
    expected = N_TISSUES * len(thresholds) * N_PREDICTIONS
    actual = int(_method_counts.loc[method_name])
    assert actual == expected, (
        f'{method_name}: expected {expected}, got {actual} -- '
        f'{_METHOD_SOURCE_NB[method_name]} must be complete '
        f'({N_TISSUES} tissues x {len(thresholds)} thresholds)')

method
gene_based               167825
module_based_archs4      167825
module_based_gtex        167825
module_based_recount2    167825
Name: count, dtype: int64

'Unique drug-disease pairs: 685'

In [11]:
_tmp = predictions.groupby(['method', 'n_top_genes'], observed=True).size().rename('n')
display(_tmp)

for method_name, thresholds in METHOD_THRESHOLDS.items():
    method_counts = _tmp.loc[method_name]
    actual_thresholds = sorted(float(x) for x in method_counts.index)
    assert actual_thresholds == sorted(thresholds), (
        f'{method_name}: expected thresholds {sorted(thresholds)}, '
        f'got {actual_thresholds}')
    assert np.all(method_counts.values == N_TISSUES * N_PREDICTIONS), method_name

method                 n_top_genes
gene_based             -1.0           33565
                        50.0          33565
                        100.0         33565
                        250.0         33565
                        500.0         33565
module_based_archs4    -1.0           33565
                        5.0           33565
                        10.0          33565
                        25.0          33565
                        50.0          33565
module_based_gtex      -1.0           33565
                        5.0           33565
                        10.0          33565
                        25.0          33565
                        50.0          33565
module_based_recount2  -1.0           33565
                        5.0           33565
                        10.0          33565
                        25.0          33565
                        50.0          33565
Name: n, dtype: int64

# Aggregate predictions

1. Average ranks across `n_top_genes` thresholds (per trait, drug, method, tissue).
2. Take the max across tissues (per trait, drug, method).

Matches the PhenoPlier / NB10 aggregation exactly.

In [12]:
def _reduce_mean(x):
    return pd.Series({
        'score': x['score'].mean(),
        'true_class': x['true_class'].unique()[0],
    })


def _reduce_max(x):
    return pd.Series({
        'score': x['score'].max(),
        'true_class': x['true_class'].unique()[0],
    })

In [13]:
predictions_avg = (
    predictions
    .groupby(['trait', 'drug', 'method', 'tissue'], observed=True)
    .apply(_reduce_mean, include_groups=False)
    .dropna()
    .groupby(['trait', 'drug', 'method'], observed=True)
    .apply(_reduce_max, include_groups=False)
    .dropna()
    .sort_index()
    .reset_index()
)

In [14]:
display(predictions_avg.shape)
display(predictions_avg.head())

assert predictions_avg.shape[0] == len(EXPECTED_METHODS) * N_PREDICTIONS
assert predictions_avg.dropna().shape == predictions_avg.shape

# AUROC per method (sanity vs NB10:
# gene ~0.583, archs4 ~0.625, gtex ~0.602, recount2 ~0.612).
from sklearn.metrics import roc_auc_score
display(
    predictions_avg.groupby('method', observed=True)
    .apply(lambda x: roc_auc_score(x['true_class'], x['score']),
           include_groups=False)
    .rename('AUROC'))

(2740, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,gene_based,316134.3,1.0
1,DOID:0050741,DB00215,module_based_archs4,324870.1,1.0
2,DOID:0050741,DB00215,module_based_gtex,349350.5,1.0
3,DOID:0050741,DB00215,module_based_recount2,398655.9,1.0
4,DOID:0050741,DB00704,gene_based,387103.6,1.0


method
gene_based               0.583382
module_based_archs4      0.625419
module_based_gtex        0.602484
module_based_recount2    0.612267
Name: AUROC, dtype: float64

## Save paired predictions

In [15]:
output_file = OUTPUT_DIR / 'predictions_paired.pkl'
display(output_file)
predictions_avg.to_pickle(output_file)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/14_signif_test/predictions_paired.pkl')